In [1]:
# Load the model 

from sentence_transformers import SentenceTransformer

model = SentenceTransformer("intfloat/multilingual-e5-small")

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

c:\Users\Mazen ISMAIL\Documents\Portfolio_Mazen\AI_bank_advisor\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Mazen ISMAIL\.cache\huggingface\hub\models--intfloat--multilingual-e5-small. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


README.md:   0%|          | 0.00/498k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

...

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

In [2]:
# Embed a question and three passages 

passages = [
    "passage: En cas de perte ou de vol de votre carte, faites opposition immédiatement.",
    "passage: Relevé de compte mensuel, en ligne ou papier : gratuit.",
    "passage: Tenue de compte pour les clients âgés de moins de 25 ans : gratuit.",
]
query = "query: on m'a volé ma carte, je fais quoi ?"

passage_vectors = model.encode(passages, normalize_embeddings=True)
query_vector = model.encode(query, normalize_embeddings=True)

print("Passages:", passage_vectors.shape)
print("Query:", query_vector.shape)
print(query_vector[:5])

Passages: (3, 384)
Query: (384,)
[ 0.05599596 -0.036778   -0.03483995 -0.05859696  0.08139243]


In [3]:
# Compare them 

scores = passage_vectors @ query_vector

for passage, score in zip(passages, scores):
    print(f"{score:.3f}  {passage}")

0.846  passage: En cas de perte ou de vol de votre carte, faites opposition immédiatement.
0.836  passage: Relevé de compte mensuel, en ligne ou papier : gratuit.
0.820  passage: Tenue de compte pour les clients âgés de moins de 25 ans : gratuit.


In [4]:
passages_no_prefix = [p.removeprefix("passage: ") for p in passages]
query_no_prefix = query.removeprefix("query: ")

passage_vectors_raw = model.encode(passages_no_prefix, normalize_embeddings=True)
query_vector_raw = model.encode(query_no_prefix, normalize_embeddings=True)

scores_raw = passage_vectors_raw @ query_vector_raw

for passage, score in zip(passages_no_prefix, scores_raw):
    print(f"{score:.3f}  {passage}")

0.863  En cas de perte ou de vol de votre carte, faites opposition immédiatement.
0.849  Relevé de compte mensuel, en ligne ou papier : gratuit.
0.826  Tenue de compte pour les clients âgés de moins de 25 ans : gratuit.


In [1]:
# Embed all the chunks 

import sys

sys.path.append("..")

import numpy as np

from src.chunker import chunk_documents
from src.embeddings import E5Embeddings
from src.loader import load_all_documents

chunks = chunk_documents(load_all_documents())
embeddings = E5Embeddings()

chunk_vectors = np.array(embeddings.embed_documents([chunk.page_content for chunk in chunks]))
print(chunk_vectors.shape)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

(481, 384)


In [2]:
# Our own search engine 

def search(question: str, k: int = 3) -> None:
    query_vector = np.array(embeddings.embed_query(question))
    scores = chunk_vectors @ query_vector
    best = np.argsort(scores)[::-1][:k]
    for i in best:
        chunk = chunks[i]
        print(f"{scores[i]:.3f}  {chunk.metadata['source']}  p.{chunk.metadata['page']}")
        print(chunk.page_content[:300])
        print()


search("on m'a volé ma carte, je fais quoi ?")

0.842  conditions_fonctionnement_cartes_2026.pdf  p.10
Document : Conditions de fonctionnement des cartes (Contrat carte) (Édition avril 2026)

ou
 › l’exécution des transactions financières et des opérations de paiement demandées par le Titulaire de la Carte ;
  - à des autorités financières, fiscales, administratives, pénales ou judiciaires, ou locale

0.841  conditions_fonctionnement_cartes_2026.pdf  p.5
Document : Conditions de fonctionnement des cartes (Contrat carte) (Édition avril 2026)

Sans 
autre action préalable pour la Carte Virtuelle ou après activation de la nouvelle Carte Physique par son Titulaire lors d’un retrait ou paiement 
avec frappe du Code sur le clavier d’un DAB/GAB ou d’un Équ

0.840  conditions_fonctionnement_cartes_2026.pdf  p.9
Document : Conditions de fonctionnement des cartes (Contrat carte) (Édition avril 2026)

15.6 Cette décision de blocage est motivée et notifiée au Titulaire de la Carte et/ou du Compte par tout moyen.
15.7 Dans ces cas, BNP Paribas p

In [4]:
# Where are the right chunks ? 

def rank_of(question: str, keyword: str) -> None:
    query_vector = np.array(embeddings.embed_query(question))
    scores = chunk_vectors @ query_vector
    order = np.argsort(scores)[::-1]
    for rank, i in enumerate(order, start=1):
        chunk = chunks[i]
        if keyword.lower() in chunk.page_content.lower():
            print(f"rank {rank:3d}  {scores[i]:.3f}  {chunk.metadata['source']}  p.{chunk.metadata['page']}")


rank_of("on m'a volé ma carte, je fais quoi ?", "opposition")

rank   2  0.841  conditions_fonctionnement_cartes_2026.pdf  p.5
rank   7  0.836  conditions_fonctionnement_cartes_2026.pdf  p.8
rank   8  0.836  tarifs_particuliers_2026.pdf  p.8
rank  10  0.835  tarifs_particuliers_2026.pdf  p.20
rank  12  0.834  conditions_fonctionnement_cartes_2026.pdf  p.9
rank  17  0.833  tarifs_particuliers_2026.pdf  p.20
rank  18  0.833  conditions_fonctionnement_cartes_2026.pdf  p.7
rank  25  0.830  conditions_fonctionnement_cartes_2026.pdf  p.10
rank  31  0.828  conditions_fonctionnement_cartes_2026.pdf  p.8
rank  39  0.827  tarifs_particuliers_2026.pdf  p.18
rank  48  0.825  conditions_fonctionnement_cartes_2026.pdf  p.5
rank  52  0.825  conditions_fonctionnement_cartes_2026.pdf  p.10
rank  61  0.824  conditions_fonctionnement_cartes_2026.pdf  p.14
rank  72  0.822  conditions_fonctionnement_cartes_2026.pdf  p.4
rank  84  0.821  conditions_fonctionnement_cartes_2026.pdf  p.11
rank  90  0.820  conditions_fonctionnement_cartes_2026.pdf  p.5
rank 106  0.818  cond

In [5]:
# The formal question

formal_question = "Que faire en cas de perte ou de vol de la carte bancaire ?"

rank_of(formal_question, "opposition")
print("=" * 80)
search(formal_question)

rank   1  0.872  conditions_fonctionnement_cartes_2026.pdf  p.8
rank   2  0.868  conditions_fonctionnement_cartes_2026.pdf  p.7
rank   3  0.866  conditions_generales_bienvenue_2026.pdf  p.6
rank   7  0.861  conditions_fonctionnement_cartes_2026.pdf  p.5
rank   8  0.860  tarifs_particuliers_2026.pdf  p.8
rank  11  0.856  conditions_fonctionnement_cartes_2026.pdf  p.9
rank  13  0.854  tarifs_particuliers_2026.pdf  p.18
rank  25  0.851  conditions_fonctionnement_cartes_2026.pdf  p.6
rank  36  0.849  tarifs_particuliers_2026.pdf  p.20
rank  37  0.849  tarifs_particuliers_2026.pdf  p.20
rank  40  0.848  conditions_fonctionnement_cartes_2026.pdf  p.5
rank  45  0.847  conditions_fonctionnement_cartes_2026.pdf  p.7
rank  47  0.847  conditions_fonctionnement_cartes_2026.pdf  p.8
rank  54  0.846  conditions_fonctionnement_cartes_2026.pdf  p.7
rank  64  0.845  conditions_fonctionnement_cartes_2026.pdf  p.14
rank  76  0.842  conditions_fonctionnement_cartes_2026.pdf  p.10
rank  98  0.841  conditio

In [6]:
# A better diagnostic 

def rank_of(question: str, keyword: str) -> None:
    query_vector = np.array(embeddings.embed_query(question))
    scores = chunk_vectors @ query_vector
    order = np.argsort(scores)[::-1]
    for rank, i in enumerate(order, start=1):
        text = chunks[i].page_content
        position = text.lower().find(keyword.lower())
        if position != -1:
            snippet = text[max(0, position - 80) : position + 80].replace("\n", " ")
            print(f"rank {rank:3d}  {scores[i]:.3f}  {chunks[i].metadata['source']}  p.{chunks[i].metadata['page']}")
            print(f"          ...{snippet}...")


rank_of(formal_question, "blocage")
print("=" * 80)
rank_of("on m'a volé ma carte, je fais quoi ?", "blocage")

rank   1  0.872  conditions_fonctionnement_cartes_2026.pdf  p.8
          ...12.2 Opérations non autorisées, effectuées avant la demande d’opposition (ou de blocage) Ces dispositions ne concernent pas la Carte Virtuelle. Les opérations c...
rank   2  0.868  conditions_fonctionnement_cartes_2026.pdf  p.7
          ... carte) (Édition avril 2026)  XI – RECEVABILITÉ DES DEMANDES D’OPPOSITION OU DE BLOCAGE Pour l’exécution du Contrat, l’information ci-dessus visée « de blocage ...
rank   8  0.860  tarifs_particuliers_2026.pdf  p.8
          ... Garmin (4) Gratuit Autres presta tions (par opération) •   Frais d’opposition (blocage) de la carte par la banque Gratuit •   Réédition du code secret Gratuit ...
rank  11  0.856  conditions_fonctionnement_cartes_2026.pdf  p.9
          ... et des données qui y sont liées, survenue avant la demande d’opposition (ou de blocage) conformément à l’article  12.2  ;   ● du montant de l’opération contest...
rank  13  0.854  tarifs_particuliers_2026.pdf  p.

In [10]:
# Is the card procedure in the agreement ? 

import re

for chunk in chunks:
    if chunk.metadata["doc_type"] != "account_agreement":
        continue
    text = chunk.page_content.lower()
    if "carte" in text and (re.search(r"\bvol\b", text) or "perte" in text):
        print(f"--- p.{chunk.metadata['page']} ---")
        print(chunk.page_content[:600])
        print()

print("=" * 80)

for chunk in chunks:
    if chunk.metadata["doc_type"] == "account_agreement" and chunk.metadata["page"] == 18:
        print(chunk.page_content)

Document : Conditions générales de la convention de compte Bienvenue (Édition avril 2026)

18
II - AUTRES ANNEXES
Les autres annexes de la Convention sont :
  ●  « Conditions de fonctionnement des cartes » (le « Contrat carte ») le cas échéant,
  ●  « Les Services en ligne » incluant l’annexe « Liste des documents en ligne »,
  ●  « Guide des conditions et tarifs pour les particuliers ».


In [9]:
colloquial_question = "on m'a volé ma carte, je fais quoi ?"
formal_question = "Que faire en cas de perte ou de vol de la carte bancaire ?"

for question in [colloquial_question, formal_question]:
    print(f"########## {question}")
    rank_of(question, "perte ou du vol de la Carte, de son détournement")
    rank_of(question, "40 14 44 00")
    print()

search(colloquial_question, k=5)

########## on m'a volé ma carte, je fais quoi ?
rank  12  0.802  conditions_fonctionnement_cartes_2026.pdf  p.7
          ...tre désignée par le terme « d’opposition ». 11.1 Dès qu’il a connaissance de la perte ou du vol de la Carte, de son détournement ou de toute utilisation fraudul...
rank  58  0.795  conditions_fonctionnement_cartes_2026.pdf  p.7
          ...rte VISA Classic, Carte BNP Net, Carte VISA Plus et Carte Virtuelle : +33 (0) 1 40 14 44 00.   - Carte VISA Premier : +33 (0)1 40 14 10 10.   - Carte VISA Infin...

########## Que faire en cas de perte ou de vol de la carte bancaire ?
rank   7  0.837  conditions_fonctionnement_cartes_2026.pdf  p.7
          ...tre désignée par le terme « d’opposition ». 11.1 Dès qu’il a connaissance de la perte ou du vol de la Carte, de son détournement ou de toute utilisation fraudul...
rank  32  0.826  conditions_fonctionnement_cartes_2026.pdf  p.7
          ...rte VISA Classic, Carte BNP Net, Carte VISA Plus et Carte Virtuelle : +33 (0) 1 4

## Experiment log: retrieval of the stolen-card procedure

Targets: card contract, article 11.1 (what to do) and 11.2 (phone numbers).
Ranks out of all chunks.

| # | Change | Chunks | Colloquial 11.1 | Colloquial 11.2 | Formal 11.1 | Formal 11.2 |
|---|--------|--------|-----------------|-----------------|-------------|-------------|
| 1 | Card contract added (e5-small) | 506 | 27 | 130 | 2 | 54 |
| 2 | Layout noise removed, short chunks merged (e5-small) | 481 | 18 | 107 | 2 | 54 |
| 3 | Bigger model: multilingual-e5-base | 481 | 12 | 58 | 7 | 32 |

## Experiment log: retrieval test set (23 questions)

From now on, this table decides. Hit@5 = share of questions with a correct chunk in the top 5.

| # | Change | Hit@5 | MRR |
|---|--------|-------|-----|
| A | multilingual-e5-small | 65% | 0.613 |
| B | multilingual-e5-base | 87% | 0.619 |
| B' | multilingual-e5-base, test fixed (dispute question: card contract chapter XVI added as a valid target) | ? | ? |

In [13]:
search("J'ai combien de temps pour contester une opération que je n'ai pas faite ?", k=5)

0.831  conditions_fonctionnement_cartes_2026.pdf  p.9
Document : Conditions de fonctionnement des cartes (Contrat carte) (Édition avril 2026)

XVI – CONTESTATIONS
16.1 Le Titulaire de la Carte et/ou du Compte (ou son Représentant légal, le cas échéant) a la possibilité de contester une opération, si possible 
et selon le type de Carte en présentant le 

0.825  conditions_fonctionnement_cartes_2026.pdf  p.7
Document : Conditions de fonctionnement des cartes (Contrat carte) (Édition avril 2026)

11.3 Un numéro d’enregistrement de cette demande d’opposition (ou de blocage) est communiqué au Titulaire de la Carte et/ou du Compte.  
Une trace de cette opposition (ou blocage) est conservée pendant 18 mois p

0.825  conditions_fonctionnement_cartes_2026.pdf  p.9
Document : Conditions de fonctionnement des cartes (Contrat carte) (Édition avril 2026)

XVII – REMBOURSEMENT DES OPÉRATIONS NON AUTORISÉES OU MAL EXÉCUTÉES
17.1 Opération de paiement non autorisée
Le Titulaire de la Carte et/ou du Co

In [14]:
for chunk in chunks:
    if "16.1 Le Titulaire de la Carte" in chunk.page_content:
        print(chunk.page_content)

Document : Conditions de fonctionnement des cartes (Contrat carte) (Édition avril 2026)

XVI – CONTESTATIONS
16.1 Le Titulaire de la Carte et/ou du Compte (ou son Représentant légal, le cas échéant) a la possibilité de contester une opération, si possible 
et selon le type de Carte en présentant le ticket émis par l’Équipement Électronique ou l’automate de pièces de monnaie BNP Paribas ou Cash 
Services ou le DAB/GAB ou un justificatif de l’ordre de paiement sur lequel porte un litige, sans tarder et dans un délai maximum de 13 mois à 
compter de la date du débit de l’ordre de paiement contesté.
Le délai maximum durant lequel le Titulaire de la Carte et/ou du Compte (ou son Représentant légal, le cas échéant) a la possibilité de contester 
une opération, est fixé à 70 jours à compter de la date du débit de l’ordre de paiement contesté, lorsque le prestataire de service de paiement de 
l’Accepteur est situé hors de l’EEE, de Saint-Pierre-et-Miquelon ou de Saint-Barthélemy.


In [8]:
embeddings = E5Embeddings("intfloat/multilingual-e5-base")
chunk_vectors = np.array(embeddings.embed_documents([chunk.page_content for chunk in chunks]))
print(chunk_vectors.shape)

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

c:\Users\Mazen ISMAIL\Documents\Portfolio_Mazen\AI_bank_advisor\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Mazen ISMAIL\.cache\huggingface\hub\models--intfloat--multilingual-e5-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


README.md:   0%|          | 0.00/179k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

(481, 768)


In [17]:
import json

with open("../data/eval/retrieval_set.json", encoding="utf-8") as f:
    retrieval_set = json.load(f)

print("Questions:", len(retrieval_set))

for item in retrieval_set:
    for target in item["targets"]:
        if not any(target in chunk.page_content for chunk in chunks):
            print("Target not found:", target)

Questions: 23


In [18]:
print(retrieval_set[21])

{'question': "J'ai combien de temps pour contester une opération que je n'ai pas faite ?", 'targets': ['au plus tard dans les 13 mois', '16.1 Le Titulaire de la Carte']}


In [19]:
# The evaluation function 

def first_target_rank(question, targets, emb, vectors):
    query_vector = np.array(emb.embed_query(question))
    order = np.argsort(vectors @ query_vector)[::-1]
    for rank, i in enumerate(order, start=1):
        if any(target in chunks[i].page_content for target in targets):
            return rank


def evaluate_retrieval(emb, vectors, k=5):
    ranks = [first_target_rank(item["question"], item["targets"], emb, vectors) for item in retrieval_set]
    for item, rank in zip(retrieval_set, ranks):
        print(f"{rank:4d}  {item['question']}")
    hit_rate = sum(rank <= k for rank in ranks) / len(ranks)
    mrr = sum(1 / rank for rank in ranks) / len(ranks)
    print(f"\nHit@{k}: {hit_rate:.0%}   MRR: {mrr:.3f}")


evaluate_retrieval(embeddings, chunk_vectors)

   2  Combien coûte la tenue de compte par mois ?
   3  J'habite aux États-Unis, combien je paie pour la tenue de compte ?
   4  J'ai 22 ans, est-ce que je paie des frais de tenue de compte ?
   3  Combien coûte l'option Reflets ?
   3  How much does the Reflets option cost?
   1  Je suis malvoyant, je peux avoir mes relevés en braille ?
   1  C'est quoi le prix du relevé Panorama ?
   1  Je peux encore utiliser Facilidom pour changer de banque ?
   4  Fermer mon compte, ça coûte combien ?
   3  Je déménage, je peux transférer mon compte dans une autre agence ?
   8  Vos tarifs sont-ils TTC ?
   1  Comment je dépose des billets sur mon compte ?
   1  Un chèque est valable combien de temps ?
   3  J'ai perdu mon chéquier, je fais quoi ?
   1  Le chèque que j'ai déposé est revenu impayé, que se passe-t-il ?
  12  on m'a volé ma carte, je fais quoi ?
   1  Quel numéro appeler pour bloquer ma Visa Premier ?
   1  Le voleur a utilisé ma carte avant que je la bloque, combien je vais perdre ?

In [20]:
# Compare with the small model 

small = E5Embeddings("intfloat/multilingual-e5-small")
small_vectors = np.array(small.embed_documents([chunk.page_content for chunk in chunks]))
evaluate_retrieval(small, small_vectors)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

   1  Combien coûte la tenue de compte par mois ?
   1  J'habite aux États-Unis, combien je paie pour la tenue de compte ?
   1  J'ai 22 ans, est-ce que je paie des frais de tenue de compte ?
   3  Combien coûte l'option Reflets ?
   3  How much does the Reflets option cost?
   1  Je suis malvoyant, je peux avoir mes relevés en braille ?
   1  C'est quoi le prix du relevé Panorama ?
   1  Je peux encore utiliser Facilidom pour changer de banque ?
  82  Fermer mon compte, ça coûte combien ?
  10  Je déménage, je peux transférer mon compte dans une autre agence ?
  63  Vos tarifs sont-ils TTC ?
  23  Comment je dépose des billets sur mon compte ?
   1  Un chèque est valable combien de temps ?
  29  J'ai perdu mon chéquier, je fais quoi ?
   1  Le chèque que j'ai déposé est revenu impayé, que se passe-t-il ?
  18  on m'a volé ma carte, je fais quoi ?
   1  Quel numéro appeler pour bloquer ma Visa Premier ?
   7  Le voleur a utilisé ma carte avant que je la bloque, combien je vais perdre ?

In [1]:
import json
import sys

sys.path.append("..")

from src.indexer import load_index

vectorstore = load_index()
print("Chunks in the index:", len(vectorstore.get()["ids"]))

with open("../data/eval/retrieval_set.json", encoding="utf-8") as f:
    retrieval_set = json.load(f)


def chroma_rank(question, targets, max_k=100):
    results = vectorstore.similarity_search(question, k=max_k)
    for rank, doc in enumerate(results, start=1):
        if any(target in doc.page_content for target in targets):
            return rank
    return max_k + 1


ranks = [chroma_rank(item["question"], item["targets"]) for item in retrieval_set]
hit_rate = sum(rank <= 5 for rank in ranks) / len(ranks)
mrr = sum(1 / rank for rank in ranks) / len(ranks)
print(f"Hit@5: {hit_rate:.0%}   MRR: {mrr:.3f}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Chunks in the index: 481
Hit@5: 91%   MRR: 0.661
